# Modelos de IA do VisionHat

Este notebook treina três modelos simples para apoiar o VisionHat: K-Means para agrupar padrões de análise, Random Forest para classificar o nível de atenção e Regressão Linear para estimar a confiança da detecção.

O dataset é uma carga experimental documentada para o protótipo. Ele representa medidas que podem ser recebidas pela câmera e pelos sensores do óculos. Os resultados não são uma garantia de segurança e devem ser reavaliados quando houver dados reais coletados com consentimento.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DIRETORIO = Path.cwd()
if not (DIRETORIO / 'dataset' / 'dados_visionhat.csv').is_file():
    DIRETORIO = DIRETORIO / 'ia'

ARQUIVO_DADOS = DIRETORIO / 'dataset' / 'dados_visionhat.csv'
ARQUIVO_MODELOS = DIRETORIO / 'modelos_visionhat.joblib'
FEATURES = [
    'quantidade_obstaculos',
    'distancia_media_metros',
    'luminosidade',
    'velocidade_movimento',
    'tipo_obstaculo_codigo',
]
FEATURES_CLUSTER = [
    'quantidade_obstaculos',
    'distancia_media_metros',
]
COLUNA_ATENCAO = 'nivel_atencao'
COLUNA_CONFIANCA = 'confianca_deteccao'

if not ARQUIVO_DADOS.is_file():
    raise FileNotFoundError('Dataset não encontrado: ' + str(ARQUIVO_DADOS))

dados = pd.read_csv(ARQUIVO_DADOS)
colunas_necessarias = FEATURES + [COLUNA_ATENCAO, COLUNA_CONFIANCA]
for coluna in colunas_necessarias:
    if coluna not in dados.columns:
        raise ValueError('Coluna ausente no dataset: ' + coluna)

if dados[colunas_necessarias].isnull().any().any():
    raise ValueError('O dataset possui valores ausentes.')

print('Registros:', len(dados))
print('Distribuição do nível de atenção:')
print(dados[COLUNA_ATENCAO].value_counts().sort_index())
print('Primeiros registros:')
print(dados.head().to_string(index=False))

## Modelo 1 - K-Means

O K-Means agrupa registros com características numéricas semelhantes. O nível de atenção não participa do treinamento; ele é usado apenas depois para interpretar os grupos. O StandardScaler coloca as features em escalas comparáveis.

In [ ]:
escalonador = StandardScaler()
dados_escalonados = escalonador.fit_transform(dados[FEATURES_CLUSTER])
modelo_cluster = KMeans(n_clusters=3, n_init=10, random_state=42)
grupos = modelo_cluster.fit_predict(dados_escalonados)
silhueta = silhouette_score(dados_escalonados, grupos)
ari = adjusted_rand_score(dados[COLUNA_ATENCAO], grupos)

comparacao = pd.crosstab(pd.Series(grupos, name='grupo'), dados[COLUNA_ATENCAO])
mapa_niveis = {}
for grupo in comparacao.index:
    mapa_niveis[int(grupo)] = str(comparacao.loc[grupo].idxmax())

print('Silhueta:', round(silhueta, 3))
print('Adjusted Rand Index:', round(ari, 3))
print('Tabela de comparação:')
print(comparacao)
print('Nível representativo de cada grupo:', mapa_niveis)

## Modelo 2 - Random Forest

A Random Forest classifica o nível de atenção. A base é dividida em treino e teste, mantendo a proporção das três classes. A acurácia, o relatório e a matriz de confusão mostram o resultado da divisão utilizada.

In [ ]:
x_classificacao = dados[FEATURES]
y_classificacao = dados[COLUNA_ATENCAO]
x_treino, x_teste, y_treino, y_teste = train_test_split(
    x_classificacao,
    y_classificacao,
    test_size=0.2,
    random_state=42,
    stratify=y_classificacao,
)
modelo_classificacao = RandomForestClassifier(n_estimators=300, random_state=42)
modelo_classificacao.fit(x_treino, y_treino)
previsoes_classificacao = modelo_classificacao.predict(x_teste)
acuracia = accuracy_score(y_teste, previsoes_classificacao)
matriz = confusion_matrix(y_teste, previsoes_classificacao, labels=modelo_classificacao.classes_)

print('Acurácia:', round(acuracia, 3))
print('Relatório de classificação:')
print(classification_report(y_teste, previsoes_classificacao, zero_division=0))
print('Matriz de confusão:')
print(pd.DataFrame(matriz, index=modelo_classificacao.classes_, columns=modelo_classificacao.classes_))

## Modelo 3 - Regressão Linear

A Regressão Linear estima a confiança da detecção como um valor contínuo entre 0 e 100. MAE e RMSE medem o erro; R² indica quanto da variação foi explicada pela regressão. Essa estimativa é experimental e não substitui uma confiança calculada por um modelo de visão computacional real.

In [ ]:
x_regressao = dados[FEATURES]
y_regressao = dados[COLUNA_CONFIANCA]
x_reg_treino, x_reg_teste, y_reg_treino, y_reg_teste = train_test_split(
    x_regressao,
    y_regressao,
    test_size=0.2,
    random_state=42,
)
modelo_regressao = LinearRegression()
modelo_regressao.fit(x_reg_treino, y_reg_treino)
previsoes_regressao = modelo_regressao.predict(x_reg_teste)
mae = mean_absolute_error(y_reg_teste, previsoes_regressao)
rmse = mean_squared_error(y_reg_teste, previsoes_regressao) ** 0.5
r2 = r2_score(y_reg_teste, previsoes_regressao)

print('MAE:', round(mae, 2))
print('RMSE:', round(rmse, 2))
print('R²:', round(r2, 3))

## Exportação e limitações

O arquivo joblib reúne os modelos, o escalonador, a ordem das features e as métricas. A API Flask carrega esse pacote sem treinar novamente.

A base é experimental e não representa uma pesquisa com pessoas. As features foram escolhidas para representar entradas futuras do óculos e os rótulos foram organizados para o protótipo. Antes de usar o resultado para alertas reais, será necessário coletar dados documentados, validar os sensores e revisar os critérios de segurança.

In [ ]:
pacote_modelos = {
    'modelo_cluster': modelo_cluster,
    'escalonador': escalonador,
    'mapa_niveis': mapa_niveis,
    'modelo_classificacao': modelo_classificacao,
    'modelo_regressao': modelo_regressao,
    'features': FEATURES,
    'features_cluster': FEATURES_CLUSTER,
    'classes_atencao': [str(classe) for classe in modelo_classificacao.classes_],
    'metrica_silhueta': float(silhueta),
    'metrica_ari': float(ari),
    'metrica_acuracia': float(acuracia),
    'metrica_mae': float(mae),
    'metrica_rmse': float(rmse),
    'metrica_r2': float(r2),
}
joblib.dump(pacote_modelos, ARQUIVO_MODELOS)
print('Modelos salvos em:', ARQUIVO_MODELOS)